In [ ]:
%load_ext autoreload
%autoreload 2

In [7]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == "examples":
    package_dir = package_dir.parent
else:
    repo_candidate = Path("packages/visualization/calibrated-explanations-visualization-plotly").resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])


0

# Local Uncertainty Quadrant

This notebook demonstrates `plotly.local.uncertainty_quadrant` for a single factual local CE explanation.

The plot separates two independent dimensions:

- x-axis = absolute local impact, `abs(contribution)`
- y-axis = calibrated uncertainty width, `high - low`
- contribution sign is shown separately through the marker encoding and hover text
- high-impact / low-uncertainty points are the most reliable local drivers

The first cell installs this plugin from the local source checkout with the Plotly extra.

In [8]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import register_plotly_visualization_components
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

## Data

Generate a binary classification dataset and split it into proper training, calibration, and query data using a 60/20/20 split. The calibration set is separate from the data used to fit the model.

In [9]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=0,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=0,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.50,
    random_state=0,
    stratify=y_holdout,
)

x_proper.shape, x_cal.shape, X_query.shape

((300, 8), (100, 8), (100, 8))

## Fit and Calibrate

`WrapCalibratedExplainer` owns the model fitting and calibration sequence. The assertions make the fitted and calibrated states explicit before explanations are requested.

In [10]:
model = LogisticRegression(max_iter=1000, random_state=0)

explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

## Explain and Plot

The quadrant plot answers: which local rules are both high impact and low uncertainty? Direction is not encoded through x-position; x is always absolute impact.

In [11]:
explanations = explainer.explain_factual(X_query)
explanations[0]

Prediction [ Low ,  High]
0.171 [0.120, 0.182]
Value : Feature                                  Weight [ Low  ,  High ]
2.0   : 7 > -0.74                                -0.781 [-0.829, -0.779]
0.79  : 2 > -0.87                                -0.220 [-0.283, -0.111]
0.91  : 6 <= 1.07                                -0.164 [-0.192, -0.111]
-1.66 : 0 <= -1.44                                0.029 [ 0.018,  0.097]
-0.98 : 4 <= 1.59                                 0.019 [ 0.011,  0.060]
-1.45 : 1 <= 0.99                                -0.005 [-0.017,  0.051]

In [12]:
explanations[0].plot(style="plotly.local.uncertainty_quadrant", show=True)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.uncertainty_quadrant', 'style': 'plotly.local.uncertainty_quadrant', 'prediction': {'prediction': 0.17123287671232879, 'low': 0.12, 'high': 0.18181818181818182, 'classes': 1.0}, 'items': [{'index': 5, 'feature_index': 7, 'rule': '7 > -0.74', 'feature_name': '7', 'instance_value': 2.0048913581330936, 'contribution': -0.7809202333355181, 'absolute_impact': 0.7809202333355181, 'low': -0.8287671232876712, 'high': -0.7785025730231209, 'interval_width': 0.050264550264550345, 'crosses_zero': False, 'direction': 'negative', 'quadrant': 'robust_driver', 'status_flags': ()}, {'index': 2, 'feature_index': 2, 'rule': '2 > -0.87', 'feature_name': '2', 'instance_value': 0.785703580892626, 'contribution': -0.2200780527457396, 'absolute_impact': 0.2200780527457396, 'low': -0.28331257783312574, 'high': -0.11119136571191363, 'interval_width': 0.17212121212121212, 'crosses_zero': False, 'direction': 'negative', 'quadrant': 'uncertain_driver', 'sta